# 04 Composite Factor

Build multiple composite-factor variants, compare their IC quality, and inspect learned factor weights.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "apps").exists():
            return candidate
    raise RuntimeError("repo root not found")


REPO_ROOT = locate_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from apps.quant_platform.research.data_loader import ResearchDataLoader
from apps.quant_platform.research.factor_engine.composite import CompositeFactorBuilder
from apps.quant_platform.research.analyzer.ic_analysis import analyze_factor_ic

RESEARCH_ROOT = REPO_ROOT / "apps/quant_platform/research"
OUTPUT_ROOT = RESEARCH_ROOT / "output/notebook_composite"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
loader = ResearchDataLoader()
builder = CompositeFactorBuilder()

In [ ]:
panel = loader.prepare_panel(
    loader.load_panel(
        start_date="2024-01-01",
        end_date="2024-06-30",
        columns=[
            "ts_code", "trade_date", "open", "close", "pct_chg", "turnover_rate_f",
            "volume_ratio", "pe_ttm", "pb", "ps_ttm", "dv_ttm",
        ],
    )
)
factor_cols = ["pct_chg", "turnover_rate_f", "volume_ratio", "pe_ttm", "pb"]
panel.head()

In [ ]:
methods = ["equal_weight", "ic_weighted", "icir_weighted", "pca", "ml"]
comparison_rows = []
weights = {}

for method in methods:
    composite_panel = builder.build(panel, factor_cols=factor_cols, target_col="overnight_return", method=method)
    ic_result = analyze_factor_ic(composite_panel, factor_col="composite_factor", target_col="overnight_return")
    comparison_rows.append(
        {
            "method": method,
            "ml_model": composite_panel.attrs.get("composite_ml_model"),
            "mean_ic": ic_result["mean_ic"],
            "rank_ic": ic_result["rank_ic"],
            "ic_ir": ic_result["ic_ir"],
            "coverage": ic_result.get("coverage", 0.0),
        }
    )
    weights[method] = composite_panel.filter(regex=r"^weight_").head(1).T.rename(columns={composite_panel.index[0]: method})

comparison = pd.DataFrame(comparison_rows).sort_values("ic_ir", ascending=False)
comparison.to_csv(OUTPUT_ROOT / "04_composite_comparison.csv", index=False)
comparison

In [ ]:
weight_frame = pd.concat(weights.values(), axis=1).fillna(0.0)
weight_frame.to_csv(OUTPUT_ROOT / "04_composite_weights.csv")
weight_frame

## Selection Rule

- Start with the simplest method that clears your IC and stability thresholds.
- Prefer ML only when it improves out-of-sample quality, not just in-sample IC.
- Review `04_composite_weights.csv` for concentration risk before using a learned composite in strategy backtests.